# 02 — Verify yearly routing destinations

Check the destination products generated by `TOOLS/pois/01_build_yearly_routing_destinations.ipynb`. Inputs combine GIP motorway exits, public-transport stops and stations, higher education, and curated centres. Historical OSM POIs are not used.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import pandas as pd

PROJECT_DIR = Path(r"D:\CO2_Masterarbeit\CO2_Masterarbeit")
POI_DIR = PROJECT_DIR / "TOOLS" / "pois"
RAIL_CANDIDATE_DIR = PROJECT_DIR / "OGD" / "Public_Transport" / "Rail stations"
MUNICIPALITIES_PATH = PROJECT_DIR / "OGD" / "Gemeindegrenzen.zip"
CRS_ANALYSIS = "EPSG:3035"
CRS_ROUTING = "EPSG:4326"
YEARS = range(2015, 2026)
EXPECTED_TYPES = {"motorway_exit", "rail_station", "regional_centre", "urban_centre", "higher_education", "pt_stop"}
REQUIRED_COLUMNS = {"year", "poi_type", "poi_id", "source_file", "source_schema", "provenance", "geometry"}

sys.path.insert(0, str(POI_DIR))
from destination_builders import rail_station_candidates, yearly_transport_stops

In [ ]:
municipalities = gpd.read_file(f"zip://{MUNICIPALITIES_PATH.resolve().as_posix()}").to_crs(CRS_ANALYSIS)
study_area = gpd.GeoSeries([municipalities.union_all().buffer(5_000)], crs=CRS_ANALYSIS).to_crs(CRS_ROUTING).iloc[0]
records = []
reference_columns = reference_crs = None
for year in YEARS:
    poi_path = POI_DIR / f"austria-{year}-pois.geoparquet"
    candidate_path = RAIL_CANDIDATE_DIR / f"austria-{year}-rail_station_candidates.geoparquet"
    if not poi_path.exists() or not candidate_path.exists():
        records.append({"year": year, "status": "CHECK", "detail": "missing POI or station-candidate file"})
        continue
    pois = gpd.read_parquet(poi_path)
    candidates = gpd.read_parquet(candidate_path)
    expected_candidates = rail_station_candidates(yearly_transport_stops(PROJECT_DIR, year))
    expected_candidates = expected_candidates[expected_candidates.geometry.within(study_area)]
    columns = tuple(pois.columns)
    crs = pois.crs.to_string() if pois.crs else None
    if reference_columns is None:
        reference_columns, reference_crs = columns, crs
    checks = {
        "required schema": REQUIRED_COLUMNS.issubset(pois.columns),
        "schema consistent": columns == reference_columns,
        "CRS consistent": crs == reference_crs == "EPSG:3035",
        "year consistent": set(pois["year"].dropna().astype(int)) == {year},
        "POI types valid": set(pois["poi_type"].dropna()) == EXPECTED_TYPES,
        "geometry complete": pois.geometry.notna().all(),
        "station candidates agree": set(candidates["station_id"].astype(str)) == set(expected_candidates["station_id"].astype(str)),
    }
    for detail, passed in checks.items():
        records.append({"year": year, "status": "OK" if passed else "CHECK", "detail": detail})

verification = pd.DataFrame(records)
if (verification.status != "OK").any():
    raise AssertionError(verification[verification.status != "OK"].to_string(index=False))
verification.groupby(["detail", "status"]).size().rename("years").reset_index()